# 10 — Interactive Time-Varying Survival Copilot

This notebook brings the actuarial science of **Survival Analysis** into an interactive marketing dashboard.
Instead of predicting a static outcome, it models the **dynamic daily risk (hazard)** of a creative fatiguing, using time-varying covariates (exposure, CTR decay, spend velocity).

**How it works:**
1. You select a `kpi_goal`. The notebook dynamically subsets all creatives with this goal.
2. It trains a `CoxTimeVaryingFitter` on the fly to discover the specific fatigue drivers for that cohort.
3. You select a `creative_id` and "time travel" to a specific day.
4. Gemini Copilot translates the Cox Hazard Ratios and the creative's daily metrics into an explainable business narrative.

Set `GOOGLE_API_KEY` in your environment or paste it in the config cell before running.


In [ ]:
%pip install google-generativeai pillow ipywidgets scikit-learn shap lifelines matplotlib seaborn --quiet

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import google.generativeai as genai
from lifelines import CoxTimeVaryingFitter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../../data")
MODEL    = "gemini-3.1-flash"

API_KEY  = os.environ.get("GOOGLE_API_KEY", "")
if not API_KEY:
    API_KEY = input("Paste your Google AI Studio API key: ").strip()

genai.configure(api_key=API_KEY)
print("Client ready")


## 1. Load Data and Compute Time-Varying Features

In [ ]:
daily_raw = pd.read_csv(DATA_DIR / "creative_daily_country_os_stats.csv", parse_dates=["date"])
crs_summary = pd.read_csv(DATA_DIR / "creative_summary.csv")
campaigns = pd.read_csv(DATA_DIR / "campaigns.csv")[['campaign_id', 'kpi_goal']]

# Aggregate daily
agg_cols = {
    'spend_usd': 'sum', 'impressions': 'sum', 'clicks': 'sum', 'conversions': 'sum',
    'impressions_last_7d': 'max', 'days_since_launch': 'max',
}
daily = daily_raw.groupby(['creative_id', 'campaign_id', 'date'], as_index=False).agg(agg_cols).sort_values(['creative_id', 'days_since_launch']).reset_index(drop=True)

# Derived metrics
daily['ctr'] = daily['clicks'] / daily['impressions'].clip(lower=1)
daily['cvr'] = daily['conversions'] / daily['clicks'].clip(lower=1)

# Time-varying features
daily['log_impressions_last7d'] = np.log1p(daily['impressions_last_7d'])
daily['ctr_peak_so_far'] = daily.groupby('creative_id')['ctr'].transform(lambda x: x.expanding().max())
daily['cvr_peak_so_far'] = daily.groupby('creative_id')['cvr'].transform(lambda x: x.expanding().max())
daily['ctr_vs_peak'] = (daily['ctr'] / daily['ctr_peak_so_far'].clip(lower=1e-9)).clip(0, 1)
daily['cvr_vs_peak'] = (daily['cvr'] / daily['cvr_peak_so_far'].clip(lower=1e-9)).clip(0, 1)
daily['spend_velocity_7d'] = daily.groupby('creative_id')['spend_usd'].transform(lambda x: x.rolling(7, min_periods=1).mean())

def rolling_slope(series, window=7):
    slopes = [np.nan] * len(series)
    arr = series.values
    for i in range(len(arr)):
        start = max(0, i - window + 1)
        chunk = arr[start:i+1]
        if len(chunk) >= 2:
            x = np.arange(len(chunk), dtype=float)
            slopes[i] = float(np.polyfit(x, chunk, 1)[0])
        else:
            slopes[i] = 0.0
    return slopes

daily['ctr_7d_slope'] = daily.groupby('creative_id')['ctr'].transform(rolling_slope)
tv_cols = ['log_impressions_last7d', 'ctr_vs_peak', 'cvr_vs_peak', 'spend_velocity_7d', 'ctr_7d_slope']
daily[tv_cols] = daily[tv_cols].fillna(method='ffill').fillna(0)

# Merge labels and KPIs
label_cols = ['creative_id', 'creative_status', 'fatigue_day', 'total_days_active', 'format', 'vertical']
daily = daily.merge(crs_summary[label_cols], on='creative_id', how='left')
daily = daily.merge(campaigns, on='campaign_id', how='left')

daily['is_fatigued'] = (daily['creative_status'] == 'fatigued').astype(int)
daily['start'] = daily['days_since_launch']
daily['stop']  = daily['days_since_launch'] + 1
daily['event_on_stop'] = ((daily['is_fatigued'] == 1) & (daily['stop'] == daily['fatigue_day'])).astype(int)
daily['observed_duration'] = np.where(daily['is_fatigued'] == 1, daily['fatigue_day'], daily['total_days_active'])

daily_valid = daily[daily['stop'] <= daily['observed_duration']].copy()

print(f"Data ready. Total intervals: {len(daily_valid)}")


## 2. Interactive Copilot Widgets

In [ ]:
# Pre-encode format and vertical for modeling
df_model_base = pd.get_dummies(daily_valid, columns=['format', 'vertical'], drop_first=True, dtype=float)
format_dummies = [c for c in df_model_base.columns if c.startswith('format_')]
vertical_dummies = [c for c in df_model_base.columns if c.startswith('vertical_')]
model_cols = ['creative_id', 'start', 'stop', 'event_on_stop'] + tv_cols + format_dummies + vertical_dummies

def fit_cox_for_kpi(kpi):
    print(f"Training CoxTimeVaryingFitter for KPI: {kpi}...")
    cids = daily_valid[daily_valid['kpi_goal'] == kpi]['creative_id'].unique()
    tv_df = df_model_base[df_model_base['creative_id'].isin(cids)][model_cols].copy().fillna(0)
    
    ctvf = CoxTimeVaryingFitter(penalizer=0.01)
    ctvf.fit(tv_df, id_col='creative_id', start_col='start', stop_col='stop', event_col='event_on_stop')
    
    summary = ctvf.summary[['coef', 'exp(coef)', 'p']].copy()
    summary.columns = ['coef', 'HR', 'p']
    summary = summary.sort_values('HR', ascending=False)
    
    print(f"Model trained on {len(cids)} creatives. Hazard Ratios ready.")
    return ctvf, summary, tv_df

def plot_hazard_trajectory(ctvf, tv_df, target_cid, current_day):
    # Plot partial hazard for the target creative over time
    subset = tv_df[(tv_df['creative_id'] == target_cid) & (tv_df['stop'] <= current_day + 1)]
    if subset.empty: return
    
    # Calculate partial hazard
    features = subset.drop(columns=['creative_id', 'start', 'stop', 'event_on_stop'])
    # exp(X * beta)
    hazards = np.exp(np.dot(features.values, ctvf.params_.values))
    days = subset['stop'].values
    
    plt.figure(figsize=(10, 4))
    plt.plot(days, hazards, marker='o', color='crimson', linewidth=2, label='Daily Relative Hazard')
    plt.axhline(y=1.0, color='gray', linestyle='--', label='Baseline Risk (HR=1)')
    
    plt.title(f"Dynamic Fatigue Risk Trajectory (Creative {target_cid})")
    plt.xlabel("Days Since Launch")
    plt.ylabel("Relative Hazard (Risk Multiplier)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def copilot_survival_explain(kpi, cid, day, hr_summary, creative_current_features, current_hazard):
    row = crs_summary[crs_summary['creative_id'] == cid]
    metadata = f"Format: {row.iloc[0]['format']} | Vertical: {row.iloc[0]['vertical']}"
    
    # Format top risk drivers for the KPI
    top_drivers = hr_summary[hr_summary['HR'] > 1.05].head(3)
    driver_str = "\n".join([f"- {i}: HR={r['HR']:.2f} (increases risk)" for i, r in top_drivers.iterrows()])
    
    # Current features
    feature_str = "\n".join([f"- {k}: {v:.3f}" for k,v in creative_current_features.items() if k in tv_cols])
    
    prompt = (
        f"You are the Smadex Survival Analysis Copilot. We are evaluating Creative {cid} at Day {day} for a {kpi} campaign.\n"
        f"A Time-Varying Cox model has calculated a daily Relative Hazard score of {current_hazard:.2f} (where >1.0 means elevated risk of fatiguing).\n\n"
        f"Global Fatigue Drivers for {kpi} campaigns (Hazard Ratios > 1 increase risk):\n{driver_str}\n\n"
        f"Creative's Current Metrics on Day {day}:\n{feature_str}\n"
        f"Creative Metadata: {metadata}\n\n"
        "Explain to the marketer what the creative's current risk level means, and why it is at that level based on its current metrics and the global drivers. "
        "Keep it business-focused (e.g. 'Your risk is rising because CTR has dropped below peak while spend remains aggressive'). "
        "Do not list out raw math, synthesize it into a 3-4 sentence actionable summary."
    )
    
    model = genai.GenerativeModel(model_name=MODEL)
    response = model.generate_content(prompt)
    return response.text


In [ ]:
# Widgets
kpi_dropdown = widgets.Dropdown(options=daily_valid['kpi_goal'].dropna().unique().tolist(), description='KPI Goal:')
cid_dropdown = widgets.Dropdown(description='Creative ID:')
day_slider = widgets.IntSlider(value=7, min=1, max=30, step=1, description='Current Day:', continuous_update=False)
analyze_btn = widgets.Button(description="Assess Risk", button_style='danger')
output_area = widgets.Output()

# Global state for current model
current_kpi = None
current_ctvf = None
current_summary = None
current_tv_df = None

def update_cids(*args):
    kpi = kpi_dropdown.value
    cids = daily_valid[daily_valid['kpi_goal'] == kpi]['creative_id'].unique()
    cid_dropdown.options = cids.tolist()[:50]
kpi_dropdown.observe(update_cids, 'value')
update_cids()

def on_analyze(b):
    global current_kpi, current_ctvf, current_summary, current_tv_df
    with output_area:
        clear_output(wait=True)
        kpi = kpi_dropdown.value
        cid = cid_dropdown.value
        day = day_slider.value
        
        if kpi != current_kpi:
            current_ctvf, current_summary, current_tv_df = fit_cox_for_kpi(kpi)
            current_kpi = kpi
            
        print(f"\n--- Survival Risk Assessment: Creative {cid} at Day {day} ---")
        
        subset = current_tv_df[(current_tv_df['creative_id'] == cid) & (current_tv_df['stop'] == day+1)]
        if subset.empty:
            print("No data available for this creative on this day.")
            return
            
        features = subset.drop(columns=['creative_id', 'start', 'stop', 'event_on_stop'])
        hazard = np.exp(np.dot(features.values, current_ctvf.params_.values))[0]
        
        color = 'green' if hazard < 1.0 else ('orange' if hazard < 2.0 else 'red')
        display(HTML(f"<h2 style='color:{color};'>RELATIVE HAZARD SCORE: {hazard:.2f}x</h2>"))
        
        print("\n🤖 Copilot generating risk narrative...")
        feat_dict = features.iloc[0].to_dict()
        explanation = copilot_survival_explain(kpi, cid, day, current_summary, feat_dict, hazard)
        
        clear_output(wait=True)
        print(f"--- Survival Risk Assessment: Creative {cid} at Day {day} ---")
        display(HTML(f"<h2 style='color:{color};'>RELATIVE HAZARD SCORE: {hazard:.2f}x</h2>"))
        print("\n🤖 Copilot Explanation:")
        print(explanation)
        
        print("\n--- Hazard Trajectory ---")
        plot_hazard_trajectory(current_ctvf, current_tv_df, cid, day)

analyze_btn.on_click(on_analyze)
display(widgets.HBox([kpi_dropdown, cid_dropdown, day_slider, analyze_btn]), output_area)
